# Multi-Asset CTA Strategy V2 — Transition Strategy

## 03 — Transition Geometry, Divergence and SuperbCommand Features

This notebook builds the first **live-time transition feature set** for **Multi-Asset CTA Strategy V2 — Transition Strategy**.

Book 02 established the candidate-event population. Book 03 asks:

> At the time a candidate appears, can observable transition geometry distinguish genuine reversals from failed counter-trend moves?

All explanatory features use information available **on or before the candidate date**. Future data are used only for Book 02's ex-post labels and later evaluation.

## Feature families

### Conventional transition geometry
- distance to TSMOM, dual-MA and breakout confirmation;
- established-trend age and cumulative incumbent move;
- drawdown / recovery geometry;
- 21 / 63 / 126 / 252-observation momentum;
- fast-versus-slow disagreement.

### MACD divergence
- same-direction crossover anchors;
- price progression between anchors;
- MACD crossover-level progression;
- bullish / bearish divergence flags and continuous strength.

### SuperbCommand

The supplied SuperbCommand Pine Script is translated into a research feature engine. The notebook includes the most relevant primitive state variables from both constituent systems:

- 5-period SuperSmoother;
- 20 / 50 EMA oscillator;
- 25-period signal line;
- oscillator-signal spread;
- 2-bar oscillator and signal slopes;
- asymmetric +1σ / -2σ oscillator envelope;
- Lower-BB / Upper-BB geometry and crossings;
- 10-period ATR;
- 100-observation three-cluster volatility classification;
- 3× Adaptive SuperTrend;
- SuperTrend direction, flips and distance;
- Early-Reversal, Pullback, Profit-Taking and Standby **ingredients**;
- SuperbCommand oscillator/signal crossover divergence;
- candidate-direction alignment score.

Chart colours, labels, alerts and visual-only bookkeeping are excluded.

The Pine script's long-only position state machine is not used as a target trading policy here. Instead, its underlying conditions are exposed independently so they can be tested as transition features in both directions.

## SuperbCommand timeframe

The supplied Pine Script explicitly defines the standby window in **weekly candles**. SuperbCommand is therefore calculated on completed weekly OHLC bars. Each daily Book 02 candidate receives only the last completed weekly bar on or before the candidate date, preventing partial-week lookahead.

## Research windows

- Traditional assets: formal research from **2000-01-01**.
- Digital assets: formal research from **2017-01-01**.
- Earlier history may be retained solely for indicator warm-up.

### Divergence-state correction

This revision corrects an important persistence issue identified in the first Book 03 run.

A divergence now represents the outcome of the **latest same-direction crossover comparison**. A later same-direction crossover that does not exhibit divergence explicitly resets the latest-divergence flag to `0`.

Recency-aware states are also retained:

- daily MACD divergence active within 21 / 63 local observations;
- weekly SuperbCommand divergence active within 4 / 12 completed weeks.

This prevents a historical divergence from remaining permanently active and makes the divergence family materially more falsifiable.

### Divergence timing geometry

The canonical divergence definition now preserves three dates:

- `t1` = previous crossover of the same direction;
- `t2` = most recent crossover of that direction;
- `tc` = Book 02 candidate date.

The divergence comparison is always between the **two most recent same-direction crossover anchors, `t1` and `t2`**. The two crossovers are **not required to occur within 4 weeks, 12 weeks, 21 observations, 63 observations, or any other fixed lookback window**.

Two separate continuous timing variables are retained:

- **anchor gap** = `t2 - t1`;
- **divergence recency** = `tc - t2`.

Therefore, a divergence may have been formed from crossovers that were widely separated, while the second crossover itself may be recent relative to the candidate date.

The existing 4/12-week SuperbCommand and 21/63-observation MACD variables are retained only as **descriptive diagnostic cuts**. They are not validity rules and are not the canonical definition of divergence. Later out-of-sample modelling should primarily use the continuous recency and anchor-gap variables and determine empirically whether divergence information decays with age.

In [ ]:
print("RUNNING: V2.03 TRANSITION GEOMETRY + SUPERBCOMMAND BUILD")

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("yfinance") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])

import yfinance as yf
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Google Colab; skipping Drive mount.")

PROJECT_PARENT = Path("/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2")
V201 = PROJECT_PARENT / "v2.01"
V202 = PROJECT_PARENT / "v2.02"
V203 = PROJECT_PARENT / "v2.03"

for p in [
    V203,
    V203 / "data",
    V203 / "data" / "ohlc_cache",
    V203 / "results",
    V203 / "figures",
    V203 / "manifests",
    V203 / "config",
]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "traditional_research_start": "2000-01-01",
    "digital_assets_research_start": "2017-01-01",
    "research_end": "2025-12-31",
    "momentum_horizons": [21, 63, 126, 252],
    "drawdown_window": 126,
    "macd_fast": 12,
    "macd_slow": 26,
    "macd_signal": 9,
    "normalization_window": 126,

    # SuperbCommand Pine defaults
    "sc_smoothing_length": 5,
    "sc_fast_length": 20,
    "sc_slow_length": 50,
    "sc_signal_length": 25,
    "sc_direction_length": 2,
    "sc_bb_length": 20,
    "sc_bb_smoothing": 5,
    "sc_upper_bb_multiplier": 1.0,
    "sc_lower_bb_multiplier": 2.0,
    "sc_atr_length": 10,
    "sc_supertrend_factor": 3.0,
    "sc_training_period": 100,
    "sc_high_vol_guess": 0.75,
    "sc_mid_vol_guess": 0.50,
    "sc_low_vol_guess": 0.25,
    "sc_standby_weeks": 8,

    "sc_weekly_rule": "W-FRI",
    "ohlc_download_start": "1990-01-01",
    "ohlc_download_end": "2026-01-02",
}

with open(V203 / "config" / "v2_03_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

RUNNING: V2.03 TRANSITION GEOMETRY + SUPERBCOMMAND BUILD
Mounted at /content/drive


## 1. Locate Book 01 and Book 02 inputs

In [ ]:
def find_required_file(preferred_path, filename):
    preferred_path = Path(preferred_path)
    if preferred_path.exists():
        return preferred_path

    matches = list(PROJECT_PARENT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find required input: {filename}\n"
            f"Expected near: {preferred_path}"
        )
    if len(matches) > 1:
        print(f"WARNING: multiple matches for {filename}; using {matches[0]}")
    return matches[0]


SIGNAL_PANEL_PATH = find_required_file(
    V201 / "data" / "processed" / "v2_01_signal_prices.parquet",
    "v2_01_signal_prices.parquet",
)

UNIVERSE_PATH = find_required_file(
    V201 / "manifests" / "v2_master_universe.csv",
    "v2_master_universe.csv",
)

COVERAGE_PATH = find_required_file(
    V201 / "manifests" / "v2_01_prototype_coverage.csv",
    "v2_01_prototype_coverage.csv",
)

LABELLED_EVENTS_PATH = find_required_file(
    V202 / "results" / "labelled_transition_events.csv",
    "labelled_transition_events.csv",
)

SLOW_TSMOM_PATH = find_required_file(
    V202 / "data" / "slow_tsmom.parquet",
    "slow_tsmom.parquet",
)

SLOW_MA_PATH = find_required_file(
    V202 / "data" / "slow_dual_ma.parquet",
    "slow_dual_ma.parquet",
)

SLOW_BREAKOUT_PATH = find_required_file(
    V202 / "data" / "slow_breakout.parquet",
    "slow_breakout.parquet",
)

SLOW_ENSEMBLE_PATH = find_required_file(
    V202 / "data" / "slow_ensemble.parquet",
    "slow_ensemble.parquet",
)

FAST_ENSEMBLE_PATH = find_required_file(
    V202 / "data" / "fast_ensemble.parquet",
    "fast_ensemble.parquet",
)

ESTABLISHED_PATH = find_required_file(
    V202 / "data" / "established_slow_trend.parquet",
    "established_slow_trend.parquet",
)

print("Signal panel:", SIGNAL_PANEL_PATH)
print("Universe:", UNIVERSE_PATH)
print("Labelled events:", LABELLED_EVENTS_PATH)

Signal panel: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/data/processed/v2_01_signal_prices.parquet
Universe: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/manifests/v2_master_universe.csv
Labelled events: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.02/results/labelled_transition_events.csv


## 2. Load data

In [ ]:
prices_wide = pd.read_parquet(SIGNAL_PANEL_PATH)
universe = pd.read_csv(UNIVERSE_PATH)
coverage = pd.read_csv(COVERAGE_PATH)
events = pd.read_csv(LABELLED_EVENTS_PATH, parse_dates=["candidate_date"])

slow_tsmom_wide = pd.read_parquet(SLOW_TSMOM_PATH)
slow_ma_wide = pd.read_parquet(SLOW_MA_PATH)
slow_breakout_wide = pd.read_parquet(SLOW_BREAKOUT_PATH)
slow_ensemble_wide = pd.read_parquet(SLOW_ENSEMBLE_PATH)
fast_ensemble_wide = pd.read_parquet(FAST_ENSEMBLE_PATH)
established_wide = pd.read_parquet(ESTABLISHED_PATH)

for df in [
    prices_wide,
    slow_tsmom_wide,
    slow_ma_wide,
    slow_breakout_wide,
    slow_ensemble_wide,
    fast_ensemble_wide,
    established_wide,
]:
    df.index = pd.to_datetime(df.index)

category_map = (
    universe.drop_duplicates("market")
    .set_index("market")["category"]
    .to_dict()
)

events["category"] = events["market"].map(category_map).fillna(events.get("category", "UNKNOWN"))

print("Price panel:", prices_wide.shape, prices_wide.index.min(), "to", prices_wide.index.max())
print("Labelled events:", len(events))
print(events["label"].value_counts(dropna=False))
print(events["category"].value_counts(dropna=False))

signal_symbol_map = (
    coverage.dropna(subset=["signal_symbol"])
    .drop_duplicates("market")
    .set_index("market")["signal_symbol"]
    .to_dict()
)
signal_symbol_map.setdefault("BTC-USD", "BTC-USD")


Price panel: (10571, 53) 1990-01-01 00:00:00 to 2025-12-31 00:00:00
Labelled events: 3462
label
failed     2359
genuine    1103
Name: count, dtype: int64
category
INDICES           1176
COMMODITIES       1122
FX                 717
BONDS_RATES        416
DIGITAL_ASSETS      31
Name: count, dtype: int64


'BTC-USD'

## 3. Feature helper functions

In [ ]:
def safe_zscore(series, window):
    mean = series.rolling(window, min_periods=max(20, window // 3)).mean()
    std = series.rolling(window, min_periods=max(20, window // 3)).std()
    return (series - mean) / std.replace(0, np.nan)


def macd_features(price, fast=12, slow=26, signal=9, norm_window=126):
    p = price.dropna().astype(float)

    ema_fast = p.ewm(span=fast, adjust=False).mean()
    ema_slow = p.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    hist = macd - macd_signal

    macd_pct = macd / p
    signal_pct = macd_signal / p
    hist_pct = hist / p

    cross_up = (hist > 0) & (hist.shift(1) <= 0)
    cross_down = (hist < 0) & (hist.shift(1) >= 0)

    out = pd.DataFrame(index=p.index)
    out["macd_pct"] = macd_pct
    out["macd_signal_pct"] = signal_pct
    out["macd_hist_pct"] = hist_pct
    out["macd_hist_z"] = safe_zscore(hist_pct, norm_window)
    out["macd_cross_up"] = cross_up.astype(int)
    out["macd_cross_down"] = cross_down.astype(int)

    def observations_since(mask):
        result = np.full(len(p), np.nan)
        last = None
        for i, flag in enumerate(mask.fillna(False).values):
            if flag:
                last = i
            if last is not None:
                result[i] = i - last
        return pd.Series(result, index=p.index)

    out["obs_since_bullish_cross"] = observations_since(cross_up)
    out["obs_since_bearish_cross"] = observations_since(cross_down)

    # Same-direction crossover-to-crossover divergence.
    #
    # Critical correction:
    # every same-direction crossover now writes either 1 or 0 into the
    # latest divergence state. A later non-divergent crossover therefore
    # resets the state instead of allowing an old divergence to persist
    # indefinitely.
    for side in ["bullish", "bearish"]:
        out[f"{side}_divergence_flag"] = 0.0
        out[f"{side}_divergence_strength"] = np.nan
        out[f"{side}_cross_price_delta"] = np.nan
        out[f"{side}_cross_macd_delta"] = np.nan
        out[f"{side}_cross_anchor_gap"] = np.nan

    def populate_divergence(cross_mask, side):
        dates = p.index[cross_mask.fillna(False)]
        if len(dates) < 2:
            return

        for prev_date, curr_date in zip(dates[:-1], dates[1:]):
            p0 = p.loc[prev_date]
            p1 = p.loc[curr_date]
            m0 = macd_pct.loc[prev_date]
            m1 = macd_pct.loc[curr_date]

            if any(pd.isna(x) for x in [p0, p1, m0, m1]) or p0 == 0:
                continue

            dp = p1 / p0 - 1.0
            dm = m1 - m0

            out.loc[curr_date, f"{side}_cross_price_delta"] = dp
            out.loc[curr_date, f"{side}_cross_macd_delta"] = dm
            out.loc[curr_date, f"{side}_cross_anchor_gap"] = (
                p.index.get_loc(curr_date) - p.index.get_loc(prev_date)
            )

            if side == "bullish":
                is_div = (dp < 0) and (dm > 0)
                strength = (-dp) * max(dm, 0.0)
            else:
                is_div = (dp > 0) and (dm < 0)
                strength = dp * max(-dm, 0.0)

            out.loc[curr_date, f"{side}_divergence_flag"] = float(is_div)
            out.loc[curr_date, f"{side}_divergence_strength"] = (
                strength if is_div else 0.0
            )

    populate_divergence(cross_up, "bullish")
    populate_divergence(cross_down, "bearish")

    for side, mask in [("bullish", cross_up), ("bearish", cross_down)]:
        dates = p.index[mask.fillna(False)]

        latest_flag = pd.Series(np.nan, index=p.index)
        latest_strength = pd.Series(np.nan, index=p.index)

        if len(dates):
            latest_flag.loc[dates] = out.loc[
                dates, f"{side}_divergence_flag"
            ].astype(float)

            latest_strength.loc[dates] = out.loc[
                dates, f"{side}_divergence_strength"
            ].fillna(0.0)

        out[f"latest_{side}_divergence_flag"] = latest_flag.ffill()
        out[f"latest_{side}_divergence_strength"] = latest_strength.ffill()
        latest_anchor_gap = pd.Series(np.nan, index=p.index)
        if len(dates):
            latest_anchor_gap.loc[dates] = out.loc[
                dates, f"{side}_cross_anchor_gap"
            ]
        out[f"latest_{side}_anchor_gap"] = latest_anchor_gap.ffill()

    # Recency-aware active states.
    out["bullish_divergence_active_21"] = (
        (out["latest_bullish_divergence_flag"] == 1)
        & (out["obs_since_bullish_cross"] <= 21)
    ).astype(float)

    out["bearish_divergence_active_21"] = (
        (out["latest_bearish_divergence_flag"] == 1)
        & (out["obs_since_bearish_cross"] <= 21)
    ).astype(float)

    out["bullish_divergence_active_63"] = (
        (out["latest_bullish_divergence_flag"] == 1)
        & (out["obs_since_bullish_cross"] <= 63)
    ).astype(float)

    out["bearish_divergence_active_63"] = (
        (out["latest_bearish_divergence_flag"] == 1)
        & (out["obs_since_bearish_cross"] <= 63)
    ).astype(float)

    return out
def compute_market_features(
    price,
    slow_tsmom,
    slow_ma,
    slow_breakout,
    slow_ensemble,
    fast_ensemble,
    established,
):
    p = price.dropna().astype(float).sort_index()

    df = pd.DataFrame(index=p.index)
    df["price"] = p

    # Align stored Book 02 states to the market-local calendar.
    for name, series in {
        "slow_tsmom": slow_tsmom,
        "slow_ma": slow_ma,
        "slow_breakout": slow_breakout,
        "slow_ensemble": slow_ensemble,
        "fast_ensemble": fast_ensemble,
        "established_slow": established,
    }.items():
        df[name] = series.reindex(p.index)

    # Multi-horizon momentum.
    for h in CONFIG["momentum_horizons"]:
        df[f"ret_{h}"] = p.pct_change(h)

    # Transparent approximations to distance from slow confirmation boundaries.
    # TSMOM boundary: price equals price h observations ago.
    lag252 = p.shift(252)
    df["tsmom_distance_pct"] = p / lag252 - 1.0

    # Slow dual MA geometry.
    ma_fast = p.rolling(100, min_periods=100).mean()
    ma_slow = p.rolling(300, min_periods=300).mean()
    df["ma_spread_pct"] = (ma_fast - ma_slow) / p
    df["ma_fast_slope_21"] = ma_fast.pct_change(21)
    df["ma_slow_slope_21"] = ma_slow.pct_change(21)

    # Breakout/channel midpoint geometry.
    hh = p.rolling(252, min_periods=252).max()
    ll = p.rolling(252, min_periods=252).min()
    mid = (hh + ll) / 2.0
    df["breakout_mid_distance_pct"] = (p - mid) / p
    channel_width = (hh - ll).replace(0, np.nan)
    df["channel_position"] = (p - ll) / channel_width

    # Ensemble vote count across slow families.
    state_mat = df[["slow_tsmom", "slow_ma", "slow_breakout"]]
    df["slow_vote_sum"] = state_mat.sum(axis=1)
    df["slow_vote_abs"] = df["slow_vote_sum"].abs()

    # Fast-vs-slow disagreement.
    df["fast_slow_disagreement"] = (
        df["fast_ensemble"].fillna(0) != df["slow_ensemble"].fillna(0)
    ).astype(int)
    df["fast_countertrend_strength"] = (
        -df["fast_ensemble"] * df["established_slow"]
    )

    # Established trend age.
    est = df["established_slow"].fillna(0)
    ages = np.zeros(len(df), dtype=float)
    current_age = 0
    prev = 0
    for i, value in enumerate(est.values):
        if value == 0:
            current_age = 0
        elif value == prev:
            current_age += 1
        else:
            current_age = 1
        ages[i] = current_age
        prev = value
    df["established_trend_age"] = ages

    # Incumbent-direction cumulative move from trend-state start.
    incumbent_move = np.full(len(df), np.nan)
    anchor_price = np.nan
    prev_state = 0
    for i, (date, row) in enumerate(df.iterrows()):
        state = row["established_slow"]
        px = row["price"]
        if pd.isna(state) or state == 0:
            anchor_price = np.nan
            prev_state = 0
            continue
        if state != prev_state or np.isnan(anchor_price):
            anchor_price = px
        incumbent_move[i] = state * (px / anchor_price - 1.0)
        prev_state = state
    df["incumbent_cumulative_move"] = incumbent_move

    # Rolling drawdown/recovery geometry.
    w = CONFIG["drawdown_window"]
    roll_high = p.rolling(w, min_periods=max(20, w // 3)).max()
    roll_low = p.rolling(w, min_periods=max(20, w // 3)).min()
    df["drawdown_from_126_high"] = p / roll_high - 1.0
    df["recovery_from_126_low"] = p / roll_low - 1.0

    # Adverse move against incumbent direction.
    df["adverse_move_vs_incumbent"] = np.where(
        df["established_slow"] > 0,
        -df["drawdown_from_126_high"],
        df["recovery_from_126_low"],
    )

    # Candidate-direction versions of momentum and distances.
    candidate_direction = -df["established_slow"]
    df["candidate_direction"] = candidate_direction

    for h in CONFIG["momentum_horizons"]:
        df[f"candidate_dir_ret_{h}"] = candidate_direction * df[f"ret_{h}"]

    df["candidate_dir_tsmom_distance"] = candidate_direction * df["tsmom_distance_pct"]
    df["candidate_dir_ma_spread"] = candidate_direction * df["ma_spread_pct"]
    df["candidate_dir_breakout_distance"] = (
        candidate_direction * df["breakout_mid_distance_pct"]
    )

    # MACD features.
    macd = macd_features(
        p,
        fast=CONFIG["macd_fast"],
        slow=CONFIG["macd_slow"],
        signal=CONFIG["macd_signal"],
        norm_window=CONFIG["normalization_window"],
    )
    df = df.join(macd)

    # Direction-aware divergence features.
    df["candidate_divergence_flag"] = np.where(
        candidate_direction > 0,
        df["latest_bullish_divergence_flag"],
        df["latest_bearish_divergence_flag"],
    )
    df["candidate_divergence_strength"] = np.where(
        candidate_direction > 0,
        df["latest_bullish_divergence_strength"],
        df["latest_bearish_divergence_strength"],
    )

    df["candidate_divergence_age"] = np.where(
        candidate_direction > 0,
        df["obs_since_bullish_cross"],
        df["obs_since_bearish_cross"],
    )

    # t2 - t1: spacing between the two most recent same-direction
    # crossover anchors used to define the latest divergence state.
    df["candidate_divergence_anchor_gap"] = np.where(
        candidate_direction > 0,
        df["latest_bullish_anchor_gap"],
        df["latest_bearish_anchor_gap"],
    )

    df["candidate_divergence_active_21"] = np.where(
        candidate_direction > 0,
        df["bullish_divergence_active_21"],
        df["bearish_divergence_active_21"],
    )

    df["candidate_divergence_active_63"] = np.where(
        candidate_direction > 0,
        df["bullish_divergence_active_63"],
        df["bearish_divergence_active_63"],
    )

    return df

## 4. SuperbCommand OHLC and feature engine

V2.01 stores signal Close prices. Adaptive SuperTrend requires High, Low and Close, so this section retrieves OHLC using the signal symbol selected by Book 01.

BTC history before 2017 is deliberately retained for warm-up. The formal candidate sample still comes only from Book 02.

In [ ]:

def crossover(a, b):
    """TradingView-style crossover: a moves from <= b to > b."""
    a = pd.Series(a, index=a.index)
    b = pd.Series(b, index=a.index)
    return (a > b) & (a.shift(1) <= b.shift(1))


def crossunder(a, b):
    """TradingView-style crossunder: a moves from >= b to < b."""
    a = pd.Series(a, index=a.index)
    b = pd.Series(b, index=a.index)
    return (a < b) & (a.shift(1) >= b.shift(1))


def observations_since_event(mask):
    """Number of local observations since the latest True event."""
    mask = pd.Series(mask, index=mask.index).fillna(False).astype(bool)
    result = np.full(len(mask), np.nan)
    last = None

    for i, flag in enumerate(mask.values):
        if flag:
            last = i
        if last is not None:
            result[i] = i - last

    return pd.Series(result, index=mask.index)


def anchored_divergence(price, indicator_level, bullish_cross, bearish_cross, prefix):
    """
    Compare consecutive same-direction crossover anchors.

    Bullish divergence:
      price makes a lower crossover-anchor price while the normalized
      indicator crossover level rises.

    Bearish divergence:
      price makes a higher crossover-anchor price while the normalized
      indicator crossover level falls.

    Latest divergence fields are updated at each same-direction crossover,
    so a historical divergence does not remain permanently active.
    """
    price = pd.Series(price, dtype=float)
    indicator_level = pd.Series(indicator_level, index=price.index, dtype=float)

    out = pd.DataFrame(index=price.index)

    for side, mask in [
        ("bullish", pd.Series(bullish_cross, index=price.index).fillna(False)),
        ("bearish", pd.Series(bearish_cross, index=price.index).fillna(False)),
    ]:
        event_flag = pd.Series(0.0, index=price.index)
        strength = pd.Series(np.nan, index=price.index)
        price_delta = pd.Series(np.nan, index=price.index)
        indicator_delta = pd.Series(np.nan, index=price.index)
        anchor_gap = pd.Series(np.nan, index=price.index)

        dates = price.index[mask.astype(bool)]

        for prev_date, curr_date in zip(dates[:-1], dates[1:]):
            p0, p1 = price.loc[prev_date], price.loc[curr_date]
            i0, i1 = indicator_level.loc[prev_date], indicator_level.loc[curr_date]

            if any(pd.isna(x) for x in [p0, p1, i0, i1]) or p0 == 0:
                continue

            dp = p1 / p0 - 1.0
            di = i1 - i0

            price_delta.loc[curr_date] = dp
            indicator_delta.loc[curr_date] = di
            anchor_gap.loc[curr_date] = (
                price.index.get_loc(curr_date) - price.index.get_loc(prev_date)
            )

            if side == "bullish":
                is_div = (dp < 0) and (di > 0)
                s = (-dp) * max(di, 0.0)
            else:
                is_div = (dp > 0) and (di < 0)
                s = dp * max(-di, 0.0)

            event_flag.loc[curr_date] = float(is_div)
            strength.loc[curr_date] = s if is_div else 0.0

        latest_flag = pd.Series(np.nan, index=price.index)
        latest_strength = pd.Series(np.nan, index=price.index)

        if len(dates):
            latest_flag.loc[dates] = event_flag.loc[dates]
            latest_strength.loc[dates] = strength.loc[dates].fillna(0.0)

        out[f"{prefix}_{side}_divergence_flag"] = event_flag
        out[f"{prefix}_{side}_divergence_strength"] = strength
        out[f"{prefix}_{side}_cross_price_delta"] = price_delta
        out[f"{prefix}_{side}_cross_indicator_delta"] = indicator_delta
        out[f"{prefix}_{side}_cross_anchor_gap"] = anchor_gap
        out[f"{prefix}_latest_{side}_divergence_flag"] = latest_flag.ffill()
        out[f"{prefix}_latest_{side}_divergence_strength"] = latest_strength.ffill()
        latest_anchor_gap = pd.Series(np.nan, index=price.index)
        if len(dates):
            latest_anchor_gap.loc[dates] = anchor_gap.loc[dates]
        out[f"{prefix}_latest_{side}_anchor_gap"] = latest_anchor_gap.ffill()
        out[f"{prefix}_obs_since_{side}_cross"] = observations_since_event(mask)

    return out


def clean_yf_frame(data):
    if data is None or data.empty:
        return None
    data = data.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [c[0] for c in data.columns]
    data.index = pd.to_datetime(data.index)
    if data.index.tz is not None:
        data.index = data.index.tz_localize(None)
    data = data[~data.index.duplicated(keep="last")].sort_index()
    needed = ["Open", "High", "Low", "Close"]
    if not all(c in data.columns for c in needed):
        return None
    return data[needed].astype(float)


def download_ohlc(symbol):
    try:
        raw = yf.download(
            symbol,
            start=CONFIG["ohlc_download_start"],
            end=CONFIG["ohlc_download_end"],
            auto_adjust=False,
            progress=False,
            actions=False,
            threads=False,
        )
        return clean_yf_frame(raw)
    except Exception as exc:
        print(f"OHLC download failed for {symbol}: {exc}")
        return None


def safe_filename(text):
    return (
        str(text)
        .replace("/", "_")
        .replace("\\", "_")
        .replace("^", "IDX_")
        .replace("=", "_")
        .replace(":", "_")
    )


def resample_completed_weekly_ohlc(daily):
    x = daily[["Open", "High", "Low", "Close"]].dropna().sort_index()
    weekly = x.resample(
        CONFIG["sc_weekly_rule"],
        label="right",
        closed="right",
    ).agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
    })
    return weekly.dropna()


def pine_supersmoother(source, length):
    s = pd.Series(source, dtype=float)
    values = s.values
    result = np.full(len(s), np.nan)

    a1 = math.exp(-1.414 * math.pi / length)
    b1 = 2.0 * a1 * math.cos(1.414 * math.pi / length)
    c2 = b1
    c3 = -(a1 * a1)
    c1 = 1.0 - c2 - c3

    for i in range(len(values)):
        if np.isnan(values[i]):
            continue
        prev_src = values[i - 1] if i >= 1 and not np.isnan(values[i - 1]) else 0.0
        prev1 = result[i - 1] if i >= 1 and not np.isnan(result[i - 1]) else 0.0
        prev2 = result[i - 2] if i >= 2 and not np.isnan(result[i - 2]) else 0.0
        result[i] = c1 * (values[i] + prev_src) / 2.0 + c2 * prev1 + c3 * prev2

    return pd.Series(result, index=s.index)


def pine_rma(series, length):
    s = pd.Series(series, dtype=float)
    result = np.full(len(s), np.nan)
    vals = s.values
    valid = []

    for i, value in enumerate(vals):
        if np.isnan(value):
            continue
        if len(valid) < length:
            valid.append(value)
            if len(valid) == length:
                result[i] = np.mean(valid)
        else:
            prior_idx = np.where(~np.isnan(result[:i]))[0]
            if len(prior_idx):
                prev = result[prior_idx[-1]]
                result[i] = (prev * (length - 1) + value) / length

    return pd.Series(result, index=s.index)


def pine_atr(ohlc, length):
    prev_close = ohlc["Close"].shift(1)
    tr = pd.concat([
        ohlc["High"] - ohlc["Low"],
        (ohlc["High"] - prev_close).abs(),
        (ohlc["Low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return pine_rma(tr, length)


def rolling_three_cluster_atr(atr):
    n = CONFIG["sc_training_period"]
    vals = atr.values.astype(float)

    cluster = pd.Series(np.nan, index=atr.index)
    assigned = pd.Series(np.nan, index=atr.index)
    high_c = pd.Series(np.nan, index=atr.index)
    mid_c = pd.Series(np.nan, index=atr.index)
    low_c = pd.Series(np.nan, index=atr.index)

    for i in range(n - 1, len(vals)):
        window = vals[i - n + 1:i + 1]
        if np.isnan(window).any():
            continue

        lo = np.min(window)
        hi = np.max(window)
        span = hi - lo

        centers = np.array([
            lo + span * CONFIG["sc_high_vol_guess"],
            lo + span * CONFIG["sc_mid_vol_guess"],
            lo + span * CONFIG["sc_low_vol_guess"],
        ], dtype=float)

        for _ in range(100):
            labels = np.argmin(np.abs(window[:, None] - centers[None, :]), axis=1)
            new_centers = centers.copy()
            for k in range(3):
                members = window[labels == k]
                if len(members):
                    new_centers[k] = members.mean()

            if np.allclose(new_centers, centers, rtol=0, atol=1e-12):
                centers = new_centers
                break
            centers = new_centers

        k = int(np.argmin(np.abs(vals[i] - centers)))
        cluster.iloc[i] = k
        assigned.iloc[i] = centers[k]
        high_c.iloc[i], mid_c.iloc[i], low_c.iloc[i] = centers

    return pd.DataFrame({
        "sc_cluster_raw": cluster,
        "sc_assigned_atr_centroid": assigned,
        "sc_high_vol_centroid": high_c,
        "sc_mid_vol_centroid": mid_c,
        "sc_low_vol_centroid": low_c,
    })


def adaptive_supertrend(ohlc, adaptive_atr):
    high = ohlc["High"].values
    low = ohlc["Low"].values
    close = ohlc["Close"].values
    atr = adaptive_atr.reindex(ohlc.index).values

    n = len(ohlc)
    lower = np.full(n, np.nan)
    upper = np.full(n, np.nan)
    st = np.full(n, np.nan)
    direction = np.full(n, np.nan)

    factor = CONFIG["sc_supertrend_factor"]

    for i in range(n):
        if np.isnan(atr[i]):
            continue

        hl2 = (high[i] + low[i]) / 2.0
        raw_upper = hl2 + factor * atr[i]
        raw_lower = hl2 - factor * atr[i]

        prev_lower = lower[i - 1] if i >= 1 and not np.isnan(lower[i - 1]) else raw_lower
        prev_upper = upper[i - 1] if i >= 1 and not np.isnan(upper[i - 1]) else raw_upper
        prev_close = close[i - 1] if i >= 1 else np.nan

        lower[i] = raw_lower if (raw_lower > prev_lower) or (i >= 1 and prev_close < prev_lower) else prev_lower
        upper[i] = raw_upper if (raw_upper < prev_upper) or (i >= 1 and prev_close > prev_upper) else prev_upper

        prev_st = st[i - 1] if i >= 1 else np.nan

        if i == 0 or np.isnan(atr[i - 1]):
            direction[i] = 1
        elif np.isclose(prev_st, prev_upper, equal_nan=False):
            direction[i] = -1 if close[i] > upper[i] else 1
        else:
            direction[i] = 1 if close[i] < lower[i] else -1

        st[i] = lower[i] if direction[i] == -1 else upper[i]

    return pd.DataFrame({
        "sc_supertrend": st,
        "sc_supertrend_dir": direction,
        "sc_supertrend_upper_band": upper,
        "sc_supertrend_lower_band": lower,
    }, index=ohlc.index)


def compute_superbcommand_features(daily_ohlc):
    weekly = resample_completed_weekly_ohlc(daily_ohlc)
    close = weekly["Close"].astype(float)

    smooth = pine_supersmoother(close, CONFIG["sc_smoothing_length"])
    fast_ma = smooth.ewm(span=CONFIG["sc_fast_length"], adjust=False).mean()
    slow_ma = smooth.ewm(span=CONFIG["sc_slow_length"], adjust=False).mean()

    oscillator = fast_ma - slow_ma
    signal_line = oscillator.ewm(span=CONFIG["sc_signal_length"], adjust=False).mean()
    spread = oscillator - signal_line

    dlen = CONFIG["sc_direction_length"]
    osc_slope = oscillator - oscillator.shift(dlen)
    signal_slope = signal_line - signal_line.shift(dlen)

    raw_std = spread.rolling(
        CONFIG["sc_bb_length"],
        min_periods=CONFIG["sc_bb_length"],
    ).std(ddof=0)
    spread_std = raw_std.ewm(span=CONFIG["sc_bb_smoothing"], adjust=False).mean()

    upper_bb = signal_line + CONFIG["sc_upper_bb_multiplier"] * spread_std
    lower_bb = signal_line - CONFIG["sc_lower_bb_multiplier"] * spread_std

    atr = pine_atr(weekly, CONFIG["sc_atr_length"])
    clusters = rolling_three_cluster_atr(atr)
    st = adaptive_supertrend(weekly, clusters["sc_assigned_atr_centroid"])

    out = pd.DataFrame(index=weekly.index)
    out["sc_close"] = close
    out["sc_osc_pct"] = oscillator / close
    out["sc_signal_pct"] = signal_line / close
    out["sc_spread_pct"] = spread / close
    out["sc_osc_slope_pct"] = osc_slope / close
    out["sc_signal_slope_pct"] = signal_slope / close

    out["sc_osc_rising"] = (osc_slope > 0).astype(float)
    out["sc_osc_falling"] = (osc_slope < 0).astype(float)
    out["sc_signal_rising"] = (signal_slope > 0).astype(float)
    out["sc_signal_falling"] = (signal_slope < 0).astype(float)
    out["sc_osc_turns_red"] = ((osc_slope < 0) & ~(osc_slope.shift(1) < 0)).astype(float)

    out["sc_bb_width_pct"] = (upper_bb - lower_bb) / close
    out["sc_osc_minus_upper_bb_pct"] = (oscillator - upper_bb) / close
    out["sc_osc_minus_lower_bb_pct"] = (oscillator - lower_bb) / close
    out["sc_osc_above_upper_bb"] = (oscillator > upper_bb).astype(float)
    out["sc_osc_below_lower_bb"] = (oscillator < lower_bb).astype(float)
    out["sc_osc_between_bbs"] = ((oscillator >= lower_bb) & (oscillator <= upper_bb)).astype(float)

    zero = pd.Series(0.0, index=close.index)
    bull_cross = crossover(oscillator, signal_line)
    bear_cross = crossunder(oscillator, signal_line)

    out["sc_signal_cross_above_zero"] = crossover(signal_line, zero).astype(float)
    out["sc_osc_cross_above_signal"] = bull_cross.astype(float)
    out["sc_osc_cross_below_signal"] = bear_cross.astype(float)
    out["sc_osc_cross_above_lower_bb"] = crossover(oscillator, lower_bb).astype(float)
    out["sc_osc_cross_below_upper_bb"] = crossunder(oscillator, upper_bb).astype(float)

    out = out.join(clusters)
    out = out.join(st)

    out["sc_atr_pct"] = atr / close
    out["sc_vol_cluster"] = 3.0 - out["sc_cluster_raw"]  # 3 high, 2 medium, 1 low
    out["sc_vol_cluster_change"] = out["sc_vol_cluster"].diff()

    out["sc_supertrend_bullish"] = (out["sc_supertrend_dir"] < 0).astype(float)
    out["sc_supertrend_bearish"] = (out["sc_supertrend_dir"] > 0).astype(float)
    out["sc_supertrend_flip_bull"] = ((out["sc_supertrend_dir"] < 0) & ~(out["sc_supertrend_dir"].shift(1) < 0)).astype(float)
    out["sc_supertrend_flip_bear"] = ((out["sc_supertrend_dir"] > 0) & ~(out["sc_supertrend_dir"].shift(1) > 0)).astype(float)

    out["sc_price_minus_supertrend_atr"] = (
        (close - out["sc_supertrend"]) / atr.replace(0, np.nan)
    )

    out["sc_standby_anchor_condition"] = (
        (oscillator < lower_bb) & (out["sc_supertrend_bearish"] == 1)
    ).astype(float)

    out["sc_early_reversal_intersection_raw"] = (
        (out["sc_supertrend_bullish"] == 1)
        & (out["sc_osc_cross_above_lower_bb"] == 1)
    ).astype(float)

    out["sc_early_reversal_setup_bull"] = (
        (out["sc_supertrend_bullish"] == 1)
        & (oscillator > lower_bb)
        & (osc_slope > 0)
    ).astype(float)

    out["sc_pullback_setup_bull"] = (
        (oscillator > 0)
        & (oscillator >= lower_bb)
        & (oscillator < upper_bb)
        & (osc_slope < 0)
    ).astype(float)

    out["sc_profit_taking_turn_bull"] = (
        (oscillator > 0) & (out["sc_osc_turns_red"] == 1)
    ).astype(float)

    sc_div = anchored_divergence(
        close,
        out["sc_osc_pct"],
        bull_cross,
        bear_cross,
        prefix="sc",
    )
    return out.join(sc_div)


ohlc_by_market = {}
ohlc_status_rows = []

required_markets = sorted(set(events["market"]).intersection(prices_wide.columns))

for i, market in enumerate(required_markets, start=1):
    symbol = signal_symbol_map.get(market)
    cache_path = V203 / "data" / "ohlc_cache" / f"{safe_filename(market)}.parquet"

    data = None
    source = None

    if cache_path.exists():
        try:
            data = pd.read_parquet(cache_path)
            data.index = pd.to_datetime(data.index)
            source = "cache"
        except Exception:
            data = None

    if data is None and symbol:
        data = download_ohlc(symbol)
        source = "download"
        if data is not None and not data.empty:
            data.to_parquet(cache_path)

    if data is not None and not data.empty:
        ohlc_by_market[market] = data

    ohlc_status_rows.append({
        "market": market,
        "category": category_map.get(market, "UNKNOWN"),
        "signal_symbol": symbol,
        "ohlc_ok": data is not None and not data.empty,
        "ohlc_source": source,
        "ohlc_start": data.index.min() if data is not None and not data.empty else pd.NaT,
        "ohlc_end": data.index.max() if data is not None and not data.empty else pd.NaT,
        "ohlc_observations": len(data) if data is not None else 0,
    })

    if i % 10 == 0 or i == len(required_markets):
        print(f"OHLC processed: {i}/{len(required_markets)}")

ohlc_status = pd.DataFrame(ohlc_status_rows)
print("OHLC coverage:", f"{ohlc_status['ohlc_ok'].mean():.1%}")
display(ohlc_status)

OHLC processed: 10/53
OHLC processed: 20/53
OHLC processed: 30/53
OHLC processed: 40/53
OHLC processed: 50/53
OHLC processed: 53/53
OHLC coverage: 100.0%


,market,category,signal_symbol,ohlc_ok,ohlc_source,ohlc_start,ohlc_end,ohlc_observations
0,20+ Year Treasury ETF,BONDS_RATES,TLT,True,download,2002-07-30,2025-12-31,5895
1,30-Day Fed Funds,BONDS_RATES,ZQ=F,True,download,2000-09-01,2025-12-31,6344
2,AEX,INDICES,^AEX,True,download,1992-10-12,2025-12-31,8477
3,AUD/USD,FX,AUDUSD=X,True,download,2006-05-16,2025-12-31,5106
4,Aluminium,COMMODITIES,ALI=F,True,download,2014-05-06,2025-12-31,2896
5,BTC-USD,DIGITAL_ASSETS,BTC-USD,True,download,2014-09-17,2026-01-01,4125
6,Brent Crude Oil,COMMODITIES,BZ=F,True,download,2007-07-30,2025-12-31,4585
7,CAC 40,INDICES,^FCHI,True,download,1990-03-01,2025-12-31,9101
8,Cocoa,COMMODITIES,CC=F,True,download,2000-01-03,2025-12-31,6520
9,Coffee,COMMODITIES,KC=F,True,download,2000-01-03,2025-12-31,6518


## 4. Build market-local feature panels

In [ ]:
daily_features = {}
sc_features = {}
feature_build_rows = []

common_markets = [m for m in prices_wide.columns if m in slow_ensemble_wide.columns]

for i, market in enumerate(common_markets, start=1):
    p = prices_wide[market].dropna()
    if len(p) == 0:
        continue

    daily_features[market] = compute_market_features(
        price=p,
        slow_tsmom=slow_tsmom_wide[market],
        slow_ma=slow_ma_wide[market],
        slow_breakout=slow_breakout_wide[market],
        slow_ensemble=slow_ensemble_wide[market],
        fast_ensemble=fast_ensemble_wide[market],
        established=established_wide[market],
    )

    sc_ok = market in ohlc_by_market
    sc_error = ""
    if sc_ok:
        try:
            sc_features[market] = compute_superbcommand_features(ohlc_by_market[market])
        except Exception as exc:
            sc_error = f"{type(exc).__name__}: {exc}"
            print(f"SuperbCommand build failed for {market}: {sc_error}")
            sc_ok = False

    feature_build_rows.append({
        "market": market,
        "category": category_map.get(market, "UNKNOWN"),
        "daily_features_ok": True,
        "superbcommand_features_ok": sc_ok,
        "daily_rows": len(daily_features[market]),
        "weekly_sc_rows": len(sc_features[market]) if market in sc_features else 0,
        "superbcommand_error": sc_error,
    })

    if i % 10 == 0 or i == len(common_markets):
        print(f"Feature panels built: {i}/{len(common_markets)}")

feature_build_audit = pd.DataFrame(feature_build_rows)

print("Daily feature markets:", len(daily_features))
print("SuperbCommand feature markets:", len(sc_features))
display(feature_build_audit)

Feature panels built: 10/53
Feature panels built: 20/53
Feature panels built: 30/53
Feature panels built: 40/53
Feature panels built: 50/53
Feature panels built: 53/53
Daily feature markets: 53
SuperbCommand feature markets: 53


,market,category,daily_features_ok,superbcommand_features_ok,daily_rows,weekly_sc_rows,superbcommand_error
0,20+ Year Treasury ETF,BONDS_RATES,True,True,5895,1223,
1,30-Day Fed Funds,BONDS_RATES,True,True,6344,1323,
2,US 10Y Treasury Note,BONDS_RATES,True,True,6348,1319,
3,US 2Y Treasury Note,BONDS_RATES,True,True,6414,1336,
4,US 30Y Treasury Bond,BONDS_RATES,True,True,6354,1320,
5,US 5Y Treasury Note,BONDS_RATES,True,True,6360,1320,
6,US Ultra Treasury Bond,BONDS_RATES,True,True,4018,834,
7,Aluminium,COMMODITIES,True,True,2896,604,
8,Brent Crude Oil,COMMODITIES,True,True,4585,961,
9,Cocoa,COMMODITIES,True,True,6520,1357,


## 5. Extract candidate-date feature matrix

In [ ]:
feature_rows = []

for _, event in events.iterrows():
    market = event["market"]
    candidate_date = pd.Timestamp(event["candidate_date"])

    if market not in daily_features:
        continue

    ddf = daily_features[market]

    if candidate_date in ddf.index:
        daily_date = candidate_date
    else:
        prior = ddf.index[ddf.index <= candidate_date]
        if len(prior) == 0:
            continue
        daily_date = prior[-1]

    row = ddf.loc[daily_date].to_dict()

    row.update({
        "category": category_map.get(market, event.get("category", "UNKNOWN")),
        "market": market,
        "candidate_date": candidate_date,
        "daily_feature_date": daily_date,
        "label": event["label"],
        "sc_feature_date": pd.NaT,
        "sc_available": 0.0,
    })

    for col in [
        "confirmation_date",
        "lead_time_days",
        "incumbent_direction",
        "new_direction",
        "candidate_direction",
    ]:
        if col in event.index:
            row[f"book02_{col}"] = event[col]

    if market in sc_features:
        sdf = sc_features[market]
        valid_dates = sdf.index[sdf.index <= candidate_date]

        if len(valid_dates):
            sc_date = valid_dates[-1]
            row["sc_feature_date"] = sc_date
            row["sc_available"] = 1.0

            for col, value in sdf.loc[sc_date].items():
                row[col] = value

    feature_rows.append(row)

features = pd.DataFrame(feature_rows).sort_values(
    ["candidate_date", "market"]
).reset_index(drop=True)

# ------------------------------------------------------------------
# Robust SuperbCommand schema
# ------------------------------------------------------------------
# If OHLC retrieval or feature construction failed for every market,
# the SC columns would otherwise be absent entirely and downstream
# candidate-direction transforms would raise KeyError.
EXPECTED_SC_COLUMNS = [
    "sc_osc_pct",
    "sc_signal_pct",
    "sc_spread_pct",
    "sc_osc_slope_pct",
    "sc_signal_slope_pct",
    "sc_bb_width_pct",
    "sc_osc_minus_upper_bb_pct",
    "sc_osc_minus_lower_bb_pct",
    "sc_atr_pct",
    "sc_vol_cluster",
    "sc_vol_cluster_change",
    "sc_supertrend_bullish",
    "sc_supertrend_bearish",
    "sc_price_minus_supertrend_atr",
    "sc_standby_anchor_condition",
    "sc_early_reversal_intersection_raw",
    "sc_early_reversal_setup_bull",
    "sc_pullback_setup_bull",
    "sc_profit_taking_turn_bull",
    "sc_latest_bullish_divergence_flag",
    "sc_latest_bearish_divergence_flag",
    "sc_latest_bullish_divergence_strength",
    "sc_latest_bearish_divergence_strength",
    "sc_latest_bullish_anchor_gap",
    "sc_latest_bearish_anchor_gap",
    "sc_obs_since_bullish_cross",
    "sc_obs_since_bearish_cross",
]

for col in EXPECTED_SC_COLUMNS:
    if col not in features.columns:
        features[col] = np.nan

cand_dir = pd.to_numeric(features["candidate_direction"], errors="coerce")

features["sc_candidate_dir_osc_pct"] = cand_dir * features["sc_osc_pct"]
features["sc_candidate_dir_signal_pct"] = cand_dir * features["sc_signal_pct"]
features["sc_candidate_dir_spread_pct"] = cand_dir * features["sc_spread_pct"]
features["sc_candidate_dir_osc_slope_pct"] = cand_dir * features["sc_osc_slope_pct"]
features["sc_candidate_dir_signal_slope_pct"] = cand_dir * features["sc_signal_slope_pct"]

st_direction = np.where(
    features["sc_supertrend_bullish"] == 1,
    1.0,
    np.where(features["sc_supertrend_bearish"] == 1, -1.0, np.nan),
)
features["sc_candidate_supertrend_alignment"] = cand_dir * st_direction

features["sc_candidate_divergence_flag"] = np.where(
    cand_dir > 0,
    features["sc_latest_bullish_divergence_flag"],
    features["sc_latest_bearish_divergence_flag"],
)

features["sc_candidate_divergence_strength"] = np.where(
    cand_dir > 0,
    features["sc_latest_bullish_divergence_strength"],
    features["sc_latest_bearish_divergence_strength"],
)

features["sc_candidate_divergence_age_weeks"] = np.where(
    cand_dir > 0,
    features.get("sc_obs_since_bullish_cross"),
    features.get("sc_obs_since_bearish_cross"),
)

# t2 - t1: number of completed weekly bars between the two most
# recent same-direction crossover anchors.
features["sc_candidate_divergence_anchor_gap_weeks"] = np.where(
    cand_dir > 0,
    features.get("sc_latest_bullish_anchor_gap"),
    features.get("sc_latest_bearish_anchor_gap"),
)

features["sc_candidate_divergence_active_4w"] = (
    (features["sc_candidate_divergence_flag"] == 1)
    & (features["sc_candidate_divergence_age_weeks"] <= 4)
).astype(float)

features["sc_candidate_divergence_active_12w"] = (
    (features["sc_candidate_divergence_flag"] == 1)
    & (features["sc_candidate_divergence_age_weeks"] <= 12)
).astype(float)


features["sc_candidate_price_vs_supertrend_atr"] = (
    cand_dir * features["sc_price_minus_supertrend_atr"]
)

# Alignment score should be NaN when SuperbCommand is unavailable,
# rather than incorrectly scoring an unavailable row as zero.
alignment = pd.DataFrame({
    "osc": (features["sc_candidate_dir_osc_pct"] > 0).astype(float),
    "spread": (features["sc_candidate_dir_spread_pct"] > 0).astype(float),
    "osc_slope": (features["sc_candidate_dir_osc_slope_pct"] > 0).astype(float),
    "signal_slope": (features["sc_candidate_dir_signal_slope_pct"] > 0).astype(float),
    "supertrend": (features["sc_candidate_supertrend_alignment"] > 0).astype(float),
})

features["sc_candidate_alignment_score"] = alignment.sum(axis=1)
features.loc[features["sc_available"] != 1, "sc_candidate_alignment_score"] = np.nan

print("Feature rows:", len(features))
print("Feature columns:", len(features.columns))
print("Rows with SuperbCommand:", int(features["sc_available"].sum()))

if int(features["sc_available"].sum()) == 0:
    print(
        "WARNING: no candidate rows received SuperbCommand features. "
        "Inspect the OHLC coverage and feature-build audit printed above."
    )

display(features.head())

Feature rows: 3462
Feature columns: 153
Rows with SuperbCommand: 3462


,price,slow_tsmom,slow_ma,slow_breakout,slow_ensemble,fast_ensemble,established_slow,ret_21,ret_63,ret_126,...,sc_candidate_dir_signal_slope_pct,sc_candidate_supertrend_alignment,sc_candidate_divergence_flag,sc_candidate_divergence_strength,sc_candidate_divergence_age_weeks,sc_candidate_divergence_anchor_gap_weeks,sc_candidate_divergence_active_4w,sc_candidate_divergence_active_12w,sc_candidate_price_vs_supertrend_atr,sc_candidate_alignment_score
0,21.719999,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,0.017330,-0.182229,0.012116,...,0.000839,1.0,0.0,0.0,9.0,49.0,0.0,0.0,2.491817,4.0
1,105.580002,-1.0,-1.0,-1.0,-1.0,1.0,-1.0,0.031861,0.015192,-0.084143,...,-0.001525,-1.0,0.0,0.0,34.0,41.0,0.0,0.0,-2.535177,1.0
2,7130.899902,1.0,1.0,1.0,1.0,-1.0,1.0,-0.043192,-0.029215,0.077712,...,-0.000754,-1.0,0.0,0.0,75.0,43.0,0.0,0.0,-1.706318,1.0
3,6315.399902,1.0,1.0,-1.0,0.0,-1.0,1.0,-0.031217,-0.009240,0.056370,...,-0.000156,1.0,0.0,0.0,0.0,25.0,0.0,0.0,3.311075,3.0
4,2106.620117,1.0,1.0,1.0,1.0,-1.0,1.0,-0.075427,-0.047192,-0.013921,...,0.000164,1.0,0.0,0.0,0.0,90.0,0.0,0.0,3.510905,4.0


## 6. Data-quality and no-lookahead checks

In [ ]:
if not (features["daily_feature_date"] <= features["candidate_date"]).all():
    raise RuntimeError("Lookahead gate failed for daily features.")

sc_rows = features["sc_available"].eq(1)

if sc_rows.any():
    sc_dates = pd.to_datetime(features.loc[sc_rows, "sc_feature_date"])
    cand_dates = pd.to_datetime(features.loc[sc_rows, "candidate_date"])
    if not (sc_dates <= cand_dates).all():
        raise RuntimeError("Lookahead gate failed for SuperbCommand weekly features.")

print("No-lookahead gate: PASSED")

quality_cols = [
    "established_trend_age",
    "incumbent_cumulative_move",
    "adverse_move_vs_incumbent",
    "candidate_dir_ret_21",
    "candidate_dir_ret_63",
    "candidate_dir_ret_126",
    "candidate_dir_tsmom_distance",
    "candidate_dir_ma_spread",
    "candidate_dir_breakout_distance",
    "macd_pct",
    "macd_hist_pct",
    "candidate_divergence_flag",
    "candidate_divergence_strength",
    "candidate_divergence_age",
    "candidate_divergence_anchor_gap",
    "candidate_divergence_active_21",
    "candidate_divergence_active_63",
    "sc_osc_pct",
    "sc_signal_pct",
    "sc_spread_pct",
    "sc_osc_slope_pct",
    "sc_signal_slope_pct",
    "sc_bb_width_pct",
    "sc_osc_minus_upper_bb_pct",
    "sc_osc_minus_lower_bb_pct",
    "sc_atr_pct",
    "sc_vol_cluster",
    "sc_supertrend_bullish",
    "sc_price_minus_supertrend_atr",
    "sc_standby_anchor_condition",
    "sc_early_reversal_setup_bull",
    "sc_candidate_divergence_flag",
    "sc_candidate_divergence_strength",
    "sc_candidate_divergence_age_weeks",
    "sc_candidate_divergence_anchor_gap_weeks",
    "sc_candidate_divergence_active_4w",
    "sc_candidate_divergence_active_12w",
    "sc_candidate_alignment_score",
]

quality_rows = []

for col in quality_cols:
    if col not in features.columns:
        continue

    nonmissing = int(features[col].notna().sum())

    quality_rows.append({
        "feature": col,
        "nonmissing": nonmissing,
        "missing": int(features[col].isna().sum()),
        "coverage_rate_all_candidates": nonmissing / len(features) if len(features) else np.nan,
        "coverage_rate_sc_available_candidates": (
            features.loc[sc_rows, col].notna().mean() if sc_rows.any() else np.nan
        ),
    })

feature_quality = pd.DataFrame(quality_rows).sort_values(
    "coverage_rate_all_candidates"
)

display(feature_quality)

No-lookahead gate: PASSED


,feature,nonmissing,missing,coverage_rate_all_candidates,coverage_rate_sc_available_candidates
34,sc_candidate_divergence_anchor_gap_weeks,3384,78,0.977470,0.977470
26,sc_vol_cluster,3409,53,0.984691,0.984691
28,sc_price_minus_supertrend_atr,3409,53,0.984691,0.984691
0,established_trend_age,3462,0,1.000000,1.000000
4,candidate_dir_ret_63,3462,0,1.000000,1.000000
5,candidate_dir_ret_126,3462,0,1.000000,1.000000
2,adverse_move_vs_incumbent,3462,0,1.000000,1.000000
1,incumbent_cumulative_move,3462,0,1.000000,1.000000
8,candidate_dir_breakout_distance,3462,0,1.000000,1.000000
9,macd_pct,3462,0,1.000000,1.000000


## 7. Descriptive genuine-vs-failed diagnostics

In [ ]:
diagnostic_features = [
    "established_trend_age",
    "incumbent_cumulative_move",
    "adverse_move_vs_incumbent",
    "candidate_dir_ret_21",
    "candidate_dir_ret_63",
    "candidate_dir_ret_126",
    "candidate_dir_ret_252",
    "candidate_dir_tsmom_distance",
    "candidate_dir_ma_spread",
    "candidate_dir_breakout_distance",
    "macd_hist_z",
    "candidate_divergence_flag",
    "candidate_divergence_strength",
    "candidate_divergence_age",
    "candidate_divergence_anchor_gap",
    "candidate_divergence_active_21",
    "candidate_divergence_active_63",
    "sc_candidate_dir_osc_pct",
    "sc_candidate_dir_signal_pct",
    "sc_candidate_dir_spread_pct",
    "sc_candidate_dir_osc_slope_pct",
    "sc_candidate_dir_signal_slope_pct",
    "sc_bb_width_pct",
    "sc_atr_pct",
    "sc_vol_cluster",
    "sc_vol_cluster_change",
    "sc_candidate_supertrend_alignment",
    "sc_candidate_price_vs_supertrend_atr",
    "sc_standby_anchor_condition",
    "sc_early_reversal_setup_bull",
    "sc_pullback_setup_bull",
    "sc_profit_taking_turn_bull",
    "sc_candidate_divergence_flag",
    "sc_candidate_divergence_strength",
    "sc_candidate_divergence_age_weeks",
    "sc_candidate_divergence_anchor_gap_weeks",
    "sc_candidate_divergence_active_4w",
    "sc_candidate_divergence_active_12w",
    "sc_candidate_alignment_score",
]

summary_rows = []

for feature in diagnostic_features:
    if feature not in features.columns:
        continue

    for label, g in features.groupby("label"):
        x = pd.to_numeric(g[feature], errors="coerce").dropna()
        summary_rows.append({
            "feature": feature,
            "label": label,
            "n": len(x),
            "mean": x.mean() if len(x) else np.nan,
            "median": x.median() if len(x) else np.nan,
            "std": x.std() if len(x) else np.nan,
            "q25": x.quantile(0.25) if len(x) else np.nan,
            "q75": x.quantile(0.75) if len(x) else np.nan,
        })

feature_label_summary = pd.DataFrame(summary_rows)
display(feature_label_summary)

,feature,label,n,mean,median,std,q25,q75
0,established_trend_age,failed,2359,497.216193,349.000000,517.973653,144.000000,649.500000
1,established_trend_age,genuine,1103,517.355394,374.000000,483.975546,185.000000,661.500000
2,incumbent_cumulative_move,failed,2359,0.184037,0.044106,0.760612,-0.026412,0.199985
3,incumbent_cumulative_move,genuine,1103,0.275068,0.043335,1.137247,-0.019329,0.242682
4,adverse_move_vs_incumbent,failed,2359,0.132067,0.091231,0.155414,0.050567,0.162202
...,...,...,...,...,...,...,...,...
73,sc_candidate_divergence_active_4w,genuine,1103,0.042611,0.000000,0.202070,0.000000,0.000000
74,sc_candidate_divergence_active_12w,failed,2359,0.068249,0.000000,0.252227,0.000000,0.000000
75,sc_candidate_divergence_active_12w,genuine,1103,0.088849,0.000000,0.284654,0.000000,0.000000
76,sc_candidate_alignment_score,failed,2359,2.813904,3.000000,1.490613,1.000000,4.000000


## 8. MACD divergence event audit

In [ ]:
sc_subset = features[features["sc_available"].eq(1)].copy()

if len(sc_subset):
    sc_audit = (
        sc_subset
        .groupby(["category", "label"], dropna=False)
        .agg(
            candidates=("market", "size"),
            mean_alignment_score=("sc_candidate_alignment_score", "mean"),
            divergence_flag_rate=("sc_candidate_divergence_flag", "mean"),
            divergence_active_4w_rate=("sc_candidate_divergence_active_4w", "mean"),
            divergence_active_12w_rate=("sc_candidate_divergence_active_12w", "mean"),
            mean_divergence_strength=("sc_candidate_divergence_strength", "mean"),
            supertrend_alignment_rate=(
                "sc_candidate_supertrend_alignment",
                lambda x: (pd.to_numeric(x, errors="coerce") > 0).mean(),
            ),
            standby_anchor_rate=("sc_standby_anchor_condition", "mean"),
            mean_atr_pct=("sc_atr_pct", "mean"),
            mean_vol_cluster=("sc_vol_cluster", "mean"),
        )
        .reset_index()
    )
else:
    sc_audit = pd.DataFrame(columns=[
        "category",
        "label",
        "candidates",
        "mean_alignment_score",
        "divergence_flag_rate",
        "divergence_active_4w_rate",
        "divergence_active_12w_rate",
        "mean_divergence_strength",
        "supertrend_alignment_rate",
        "standby_anchor_rate",
        "mean_atr_pct",
        "mean_vol_cluster",
    ])
    print(
        "WARNING: SuperbCommand audit is empty because no candidate rows "
        "received valid SuperbCommand features."
    )

display(sc_audit)

btc_case = features[
    (features["market"] == "BTC-USD")
    & (features["candidate_date"] >= "2017-01-01")
    & (features["candidate_date"] <= "2018-12-31")
].copy()

if len(btc_case):
    display(btc_case)

,category,label,candidates,mean_alignment_score,divergence_flag_rate,divergence_active_4w_rate,divergence_active_12w_rate,mean_divergence_strength,supertrend_alignment_rate,standby_anchor_rate,mean_atr_pct,mean_vol_cluster
0,BONDS_RATES,failed,291,2.886598,0.185567,0.048110,0.082474,0.000250,0.481100,0.109966,0.014177,1.818815
1,BONDS_RATES,genuine,125,3.336000,0.224000,0.008000,0.008000,0.000080,0.504000,0.152000,0.012991,1.379032
2,COMMODITIES,failed,742,2.787062,0.180593,0.037736,0.064690,0.000849,0.494609,0.126685,0.059650,1.876374
3,COMMODITIES,genuine,380,3.494737,0.192105,0.034211,0.078947,0.000906,0.647368,0.100000,0.058509,1.746594
4,DIGITAL_ASSETS,failed,17,1.941176,0.588235,0.117647,0.176471,0.013150,0.352941,0.058824,0.124102,2.294118
5,DIGITAL_ASSETS,genuine,14,3.214286,0.428571,0.000000,0.214286,0.015093,0.714286,0.357143,0.169260,1.857143
6,FX,failed,457,2.855580,0.164114,0.026258,0.052516,0.000476,0.431072,0.150985,0.019504,1.700893
7,FX,genuine,260,3.034615,0.207692,0.034615,0.080769,0.000627,0.492308,0.134615,0.017670,1.532000
8,INDICES,failed,852,2.807512,0.253521,0.042254,0.072770,0.000620,0.489437,0.159624,0.056668,1.962353
9,INDICES,genuine,324,3.462963,0.271605,0.074074,0.132716,0.000502,0.617284,0.209877,0.044031,1.910494


,price,slow_tsmom,slow_ma,slow_breakout,slow_ensemble,fast_ensemble,established_slow,ret_21,ret_63,ret_126,...,sc_candidate_dir_signal_slope_pct,sc_candidate_supertrend_alignment,sc_candidate_divergence_flag,sc_candidate_divergence_strength,sc_candidate_divergence_age_weeks,sc_candidate_divergence_anchor_gap_weeks,sc_candidate_divergence_active_4w,sc_candidate_divergence_active_12w,sc_candidate_price_vs_supertrend_atr,sc_candidate_alignment_score
2250,9170.540039,1.0,1.0,-1.0,0.0,-1.0,1.0,-0.315927,-0.103879,1.196679,...,-0.026896,-1.0,0.0,0.0,155.0,NaN,0.0,0.0,-0.378425,0.0
2286,8058.669922,1.0,1.0,-1.0,0.0,-1.0,1.0,-0.018361,-0.097227,-0.524192,...,0.009957,1.0,0.0,0.0,2.0,164.0,0.0,0.0,2.128483,4.0
2301,8041.779785,1.0,1.0,-1.0,0.0,-1.0,1.0,-0.118130,-0.097795,-0.300137,...,0.014259,1.0,0.0,0.0,7.0,164.0,0.0,0.0,2.593557,4.0
2314,6582.359863,1.0,-1.0,-1.0,0.0,-1.0,1.0,-0.181480,-0.036929,-0.151101,...,0.018742,1.0,0.0,0.0,10.0,164.0,0.0,0.0,3.544057,4.0
2337,6297.569824,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,-0.183312,-0.088223,-0.069883,...,0.025846,1.0,0.0,0.0,19.0,164.0,0.0,0.0,4.129934,4.0
2354,6529.169922,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,0.030694,-0.016564,-0.329920,...,0.021486,1.0,0.0,0.0,22.0,164.0,0.0,0.0,3.736677,4.0


## 9. Save outputs

In [ ]:
output_paths = {
    "candidate_features_parquet": V203 / "data" / "v2_03_candidate_features.parquet",
    "candidate_features_csv": V203 / "results" / "v2_03_candidate_features.csv",
    "feature_quality": V203 / "results" / "v2_03_feature_quality.csv",
    "feature_label_summary": V203 / "results" / "v2_03_feature_label_summary.csv",
    "superbcommand_audit": V203 / "results" / "v2_03_superbcommand_audit.csv",
    "ohlc_coverage": V203 / "results" / "v2_03_ohlc_coverage.csv",
    "feature_build_audit": V203 / "results" / "v2_03_feature_build_audit.csv",
    "btc_case": V203 / "results" / "v2_03_btc_2017_2018_case.csv",
    "divergence_audit": V203 / "results" / "v2_03_divergence_audit.csv",
}

features.to_parquet(output_paths["candidate_features_parquet"], index=False)
features.to_csv(output_paths["candidate_features_csv"], index=False)
feature_quality.to_csv(output_paths["feature_quality"], index=False)
feature_label_summary.to_csv(output_paths["feature_label_summary"], index=False)
sc_audit.to_csv(output_paths["superbcommand_audit"], index=False)
ohlc_status.to_csv(output_paths["ohlc_coverage"], index=False)
feature_build_audit.to_csv(output_paths["feature_build_audit"], index=False)
btc_case.to_csv(output_paths["btc_case"], index=False)

divergence_audit = (
    features.groupby(["category", "label"], dropna=False)
    .agg(
        candidates=("market", "size"),

        # Daily MACD: canonical divergence state + full timing geometry
        macd_latest_flag_rate=("candidate_divergence_flag", "mean"),
        macd_mean_recency_obs=("candidate_divergence_age", "mean"),
        macd_median_recency_obs=("candidate_divergence_age", "median"),
        macd_mean_anchor_gap_obs=("candidate_divergence_anchor_gap", "mean"),
        macd_median_anchor_gap_obs=("candidate_divergence_anchor_gap", "median"),

        # These fixed windows are descriptive diagnostics only.
        macd_active_21_rate=("candidate_divergence_active_21", "mean"),
        macd_active_63_rate=("candidate_divergence_active_63", "mean"),

        # SuperbCommand: canonical divergence state + full timing geometry
        sc_latest_flag_rate=("sc_candidate_divergence_flag", "mean"),
        sc_mean_recency_weeks=("sc_candidate_divergence_age_weeks", "mean"),
        sc_median_recency_weeks=("sc_candidate_divergence_age_weeks", "median"),
        sc_mean_anchor_gap_weeks=("sc_candidate_divergence_anchor_gap_weeks", "mean"),
        sc_median_anchor_gap_weeks=("sc_candidate_divergence_anchor_gap_weeks", "median"),

        # These fixed windows are descriptive diagnostics only.
        sc_active_4w_rate=("sc_candidate_divergence_active_4w", "mean"),
        sc_active_12w_rate=("sc_candidate_divergence_active_12w", "mean"),
    )
    .reset_index()
)

divergence_audit.to_csv(output_paths["divergence_audit"], index=False)


print("Saved:")
for name, output_path in output_paths.items():
    print(f" - {name}: {output_path}")

Saved:
 - candidate_features_parquet: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/data/v2_03_candidate_features.parquet
 - candidate_features_csv: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/results/v2_03_candidate_features.csv
 - feature_quality: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/results/v2_03_feature_quality.csv
 - feature_label_summary: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/results/v2_03_feature_label_summary.csv
 - superbcommand_audit: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/results/v2_03_superbcommand_audit.csv
 - ohlc_coverage: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/results/v2_03_ohlc_coverage.csv
 - feature_build_audit: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v

# Completion Gate

Do **not** proceed to predictive classification or HMM modelling until:

- the no-lookahead gate passes;
- OHLC / SuperbCommand coverage is acceptable across major asset classes;
- core SuperbCommand features have sensible coverage after weekly warm-up;
- genuine-versus-failed distributions have been reviewed;
- MACD and SuperbCommand divergence frequencies have been inspected;
- BTC 2017–2018 has been reviewed as a transition case;
- any apparent SuperbCommand edge is benchmarked against simpler transition features.

## Outputs to upload for analysis

### Required

1. `v2.03/results/v2_03_candidate_features.csv`
2. `v2.03/results/v2_03_feature_quality.csv`
3. `v2.03/results/v2_03_feature_label_summary.csv`
4. `v2.03/results/v2_03_superbcommand_audit.csv`
5. `v2.03/results/v2_03_ohlc_coverage.csv`
6. `v2.03/results/v2_03_feature_build_audit.csv`
7. `v2.03/results/v2_03_btc_2017_2018_case.csv`
8. `v2.03/results/v2_03_divergence_audit.csv`

### Optional

9. `v2.03/data/v2_03_candidate_features.parquet`

## Research Outcome

Book 03 constructed the candidate-date information set used to distinguish genuine from failed transitions. The objective was to describe how an incumbent trend is deteriorating and how far the market has progressed toward conventional confirmation using information available at the candidate date only.

The resulting feature architecture combines:

- conventional transition geometry;
- multi-horizon momentum;
- distance to conventional trend-confirmation boundaries;
- trend age and incumbent-move information;
- drawdown and recovery structure;
- daily MACD state and divergence;
- weekly SuperbCommand state and divergence;
- price-derived volatility state;
- candidate-direction transformations.

A major conceptual development is the explicit representation of **distance to conventional confirmation**. Conventional trend rules have partially foreseeable decision boundaries: confirmation can become closer through both subsequent price movement and the passage of time as historical observations roll out of momentum, moving-average and breakout calculations. Book 03 converts this geometry into candidate-time predictors without using future prices.

The SuperbCommand implementation was reconstructed from its underlying primitives rather than importing its previous long-only trading policy. The resulting weekly variables include oscillator and signal geometry, slopes and turning behaviour, asymmetric oscillator envelopes, adaptive SuperTrend state and volatility clustering. Completed weekly observations are mapped to candidate dates to prevent partial-week look-ahead.

SuperbCommand coverage was achieved for all 53 markets and all 3,462 candidate events, with adaptive SuperTrend/volatility-state information available for approximately **98.5%** of candidates.

Descriptively, SuperbCommand already showed meaningful separation between event classes. Mean candidate-direction alignment was approximately:

\[
3.36/5 \quad \text{for genuine transitions}
\]

versus

\[
2.81/5 \quad \text{for failed transitions}.
\]

Raw early-reversal conditions occurred in approximately **25.1%** of genuine transitions versus **15.7%** of failures.

The divergence investigation produced a more nuanced result. After correcting a persistence error in the daily MACD-divergence implementation, MACD divergence proved weak and heterogeneous across markets. SuperbCommand divergence was more promising in selected areas, particularly equity indices, but divergence alone was not sufficiently strong to determine the final architecture.

### Conclusion

Book 03 shows that apparent trend transitions contain measurable candidate-time structure. In particular, **conventional confirmation geometry and SuperbCommand state variables exhibit economically coherent differences between genuine and failed transitions**, while simple oscillator divergence is substantially less reliable than initially hypothesised.

The book therefore converts the transition hypothesis into a point-in-time feature panel suitable for genuine out-of-sample classification.

**Status: FROZEN as the V2 candidate-time transition feature engine.**